In [1]:
!pip install --upgrade nbformat
!pip install nbformat ipywidgets

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px
import re
from datetime import datetime, timedelta

def magavail(start_date, end_date, mags=["pg0", "pg1"]):
    base_url = "https://themis.ssl.berkeley.edu/data/themis/thg/l2/mag/"
    all_data = []

    # Get range of years to search
    years = range(start_date.year, end_date.year + 1)

    for mag in mags:
        for year in years:
            url = f"{base_url}{mag}/{year}/"
            try:
                response = requests.get(url, timeout=10)
                if response.status_code != 200:
                    continue

                soup = BeautifulSoup(response.text, 'html.parser')
                # Pattern: thg_l2_mag_pg0_20160101_v01.cdf
                pattern = re.compile(rf"thg_l2_mag_{mag}_(\d{{8}})_v\d{{2}}\.cdf")

                for link in soup.find_all('a'):
                    match = pattern.search(link.get('href', ''))
                    if match:
                        date_str = match.group(1)
                        file_date = datetime.strptime(date_str, "%Y%m%d")

                        if start_date <= file_date <= end_date:
                            all_data.append({'Mag': mag, 'Date': file_date})
            except Exception as e:
                print(f"Error accessing {url}: {e}")

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    df = df.sort_values(['Mag', 'Date'])

    # Consolidate consecutive dates into ranges for better Gantt visualization
    df['grp'] = (df['Date'] - df.groupby('Mag')['Date'].shift(1) > timedelta(days=1)).cumsum()

    gantt_df = df.groupby(['Mag', 'grp']).agg(
        Start=('Date', 'min'),
        Finish=('Date', 'max')
    ).reset_index()

    # Add 1 day to finish to make the bar span the full day
    gantt_df['Finish'] = gantt_df['Finish'] + timedelta(days=1)

    return gantt_df

# --- Usage ---
start = datetime(2004, 1, 1)
end = datetime(2026, 12, 31)

df = magavail(start, end, mags=["pg0", "pg1", "pg2", "pg3", "pg4", "pg5"])

if not df.empty:
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Mag",
        color="Mag",
        title="AALPIP Data Availability"
    )
    # fig.update_yaxes(autorange="reversed") # Better for Gantt/Timeline
    fig.write_html("output/AALPIP Availability.html")
    fig.show()
else:
    print("No data found for the given criteria.")

In [1]:
# import pandas as pd
# import requests
# import plotly.express as px
# from datetime import datetime, timedelta
# from concurrent.futures import ThreadPoolExecutor, as_completed
#
# def check_mist_date(mag_type, station, date):
#     """
#     Checks if data exists for a specific station, type, and date on the MIST server.
#     """
#     date_str = date.strftime("%Y%m%d")
#     # MIST structure: /data/fluxgate/plot/?YMD=YYYYMMDD or /data/searchcoil/plot/?YMD=YYYYMMDD
#     url = f"http://mist.ece.vt.edu/data/{mag_type}/plot/"
#     params = {'YMD': date_str}
#
#     try:
#         # We use a HEAD request or short timeout GET to verify if the page resolves data successfully
#         response = requests.get(url, params=params, timeout=5)
#
#         # Note: Some servers return 200 but say "No Data" in text.
#         # If needed, add: if "No data available" in response.text: return None
#         if response.status_code == 200 and "Error" not in response.text:
#             return {'Mag': f"{station}_{mag_type}", 'Date': date}
#     except Exception:
#         pass
#     return None
#
# def mist_avail(start_date, end_date, mags=["pg0", "pg1"], types=["fluxgate", "searchcoil"], max_workers=20):
#     all_data = []
#
#     # Generate the list of all target dates to check
#     delta = end_date - start_date
#     date_list = [start_date + timedelta(days=i) for i in range(delta.days + 1)]
#
#     print(f"Scanning MIST server for {len(date_list)} days across {len(mags)} stations...")
#
#     # Using ThreadPoolExecutor to check dates in parallel (much faster!)
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = []
#         for mag_type in types:
#             for mag in mags:
#                 for date in date_list:
#                     futures.append(executor.submit(check_mist_date, mag_type, mag, date))
#
#         for future in as_completed(futures):
#             res = future.result()
#             if res:
#                 all_data.append(res)
#
#     if not all_data:
#         return pd.DataFrame()
#
#     df = pd.DataFrame(all_data)
#     df = df.sort_values(['Mag', 'Date'])
#
#     # Consolidate consecutive dates into ranges for Gantt visualization
#     # Grouping by 'Mag' ensures fluxgate and searchcoil variations don't bleed into each other
#     df['grp'] = (df['Date'] - df.groupby('Mag')['Date'].shift(1) > timedelta(days=1)).cumsum()
#
#     gantt_df = df.groupby(['Mag', 'grp']).agg(
#         Start=('Date', 'min'),
#         Finish=('Date', 'max')
#     ).reset_index()
#
#     # Add 1 day to finish to make the bar span the full day
#     gantt_df['Finish'] = gantt_df['Finish'] + timedelta(days=1)
#
#     return gantt_df
#
# # --- Usage ---
# # Tip: Keep range tight initially to test performance before scaling to 20 years
# start = datetime(2023, 1, 1)
# end = datetime(2024, 12, 31)
#
# # Options for types: "fluxgate", "searchcoil"
# df = mist_avail(start, end, mags=["pg0", "pg1", "pg2"], types=["fluxgate", "searchcoil"])
#
# if not df.empty:
#     fig = px.timeline(
#         df,
#         x_start="Start",
#         x_end="Finish",
#         y="Mag",
#         color="Mag",
#         title="MIST AAL-PIP Data Availability (Fluxgate & Searchcoil)"
#     )
#     fig.update_yaxes(categoryorder="category descending")
#     fig.write_html("output/MIST_Availability.html")
#     fig.show()
# else:
#     print("No data found for the given criteria.")

Scanning MIST server for 731 days across 3 stations...


In [4]:
import pandas as pd
import requests
import plotly.express as px
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

def check_mist_date(mag_type, station, date):
    """
    Checks if data exists for a specific station, type, and date on the MIST server.
    If valid, returns the base entry along with the exact URL.
    """
    date_str = date.strftime("%Y%m%d")
    url = f"http://mist.ece.vt.edu/data/{mag_type}/plot/"
    params = {'YMD': date_str}

    # Construct the full URL string for the hovertext
    full_url = f"{url}?YMD={date_str}"

    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200 and "Error" not in response.text:
            return {
                'Mag': f"{station}_{mag_type}",
                'Start': date,
                # To make a 1-day block visible in a timeline, Finish must be Day + 1
                'Finish': date + timedelta(days=1),
                'URL': full_url
            }
    except Exception:
        pass
    return None

def mist_avail_granular(start_date, end_date, mags=["pg0", "pg1"], types=["fluxgate", "searchcoil"], max_workers=20):
    all_data = []

    delta = end_date - start_date
    date_list = [start_date + timedelta(days=i) for i in range(delta.days + 1)]

    print(f"Scanning MIST server for {len(date_list)} days across {len(mags)} stations...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for mag_type in types:
            for mag in mags:
                for date in date_list:
                    futures.append(executor.submit(check_mist_date, mag_type, mag, date))

        for future in as_completed(futures):
            res = future.result()
            if res:
                all_data.append(res)

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    df = df.sort_values(['Mag', 'Start'])
    return df

# --- Usage ---
start = datetime(2000, 1, 1)
end = datetime(2024, 2, 28) # Kept tight for example speed

df = mist_avail_granular(start, end, mags=["pg0", "pg1", "pg2", "pg3", "pg4", "pg5"], types=["fluxgate", "searchcoil"])

if not df.empty:
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Mag",
        color="Mag",
        title="MIST AAL-PIP Data Availability (Granular Daily View)",
        # This includes the URL column in the interactive hover card
        hover_data={"URL": True, "Start": "|%B %d, %Y", "Finish": False}
    )

    # Optional styling: ensures distinct block gaps if you zoom in closely
    fig.update_traces(marker_line_width=1, marker_line_color="white")
    fig.update_yaxes(categoryorder="category descending")

    fig.write_html("output/MIST_Granular_Availability.html")
    fig.show()
else:
    print("No data found for the given criteria.")

Scanning MIST server for 8825 days across 6 stations...


In [3]:
df

,Mag,grp,Start,Finish
0,pg0,0,2016-01-01,2016-02-08
1,pg0,1,2016-02-09,2016-06-11
2,pg0,2,2016-10-14,2017-01-01
3,pg1,2,2016-01-01,2016-07-21
4,pg1,3,2016-10-22,2017-01-01
